In [1]:
# capstone_core.py
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from reportlab.lib import colors
from reportlab.platypus import Paragraph, Spacer, Table, TableStyle, PageBreak, SimpleDocTemplate
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.pagesizes import A4
from reportlab.lib.units import cm


def generate_report(input_csv):
    # --- Load dataset ---
    df = pd.read_csv(input_csv)

    # --- Clean and encode labels ---
    df["label_clean"] = df["label"].str.strip().str.capitalize()
    df = df[df["label_clean"].isin(["Polluted","Unpolluted"])]
    y = df["label_clean"].map({"Polluted":1, "Unpolluted":0})

    # --- Select features ---
    species_cols = [c for c in df.columns if c.startswith("P_")][:8]
    env_cols = ["pH","moisture","organic_content","temperature","nitrate"]
    X = df[species_cols + env_cols]

    # --- Train-test split ---
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, stratify=y, test_size=0.2, random_state=42
    )

    # --- Scale features ---
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # --- Train model ---
    model = RandomForestClassifier(n_estimators=200, random_state=42)
    model.fit(X_train_scaled, y_train)

    # --- Attach metadata ---
    metadata_cols = ["sample_id", "location", "date"]
    df_features = pd.DataFrame(X_test_scaled, columns=X.columns)
    df_features[metadata_cols] = df[metadata_cols].reset_index(drop=True)

    # --- Predictions ---
    probs = model.predict_proba(X_test_scaled)[:,1]
    threshold = 0.3
    df_features["Predicted_Label"] = np.where(probs >= threshold, "polluted", "unpolluted")
    df_features["Predicted_Probability"] = probs

    # --- Convert standardized to real-world values ---
    feature_stats = {
        "pH": {"mean": 6.5, "std": 0.5},
        "nitrate": {"mean": 0.8, "std": 0.5},
        "moisture": {"mean": 0.3, "std": 0.1},
        "organic_content": {"mean": 0.2, "std": 0.3},
        "temperature": {"mean": 30, "std": 5}
    }

    for feature, stats in feature_stats.items():
        if feature in df_features.columns:
            df_features[f"{feature}_real"] = df_features[feature] * stats["std"] + stats["mean"]

    # --- Suggested actions ---
    def generate_actions(row):
        actions = []
        if row["Predicted_Label"] == "polluted":
            if row["pH_real"] < 6.5:
                actions.append("Apply lime or dolomite to raise soil pH.")
            if row["nitrate_real"] > 0.8:
                actions.append("Plant nitrate-absorbing crops (e.g., barley, ryegrass).")
            if row["moisture_real"] < 0.3:
                actions.append("Increase irrigation to dilute contaminants.")
            if row["organic_content_real"] < 0.2:
                actions.append("Add compost or biochar to enhance microbial activity.")
            if row["temperature_real"] > 35:
                actions.append("Introduce shade crops or mulch to stabilize temperature.")
        else:
            actions.append("Maintain current soil conditions.")
        return "; ".join(actions)

    df_features["Suggested_Actions"] = df_features.apply(generate_actions, axis=1)

    # --- Page numbers for PDF ---
    def add_page_number(canvas, doc):
        page_num = canvas.getPageNumber()
        canvas.setFont('Helvetica', 9)
        canvas.drawRightString(19*cm, 1*cm, f"Page {page_num}")

    # --- Filter only polluted ---
    df_polluted = df_features[df_features["Predicted_Label"] == "polluted"].reset_index(drop=True)

    # --- Generate PDF ---
    styles = getSampleStyleSheet()
    elements = []
    elements.append(Paragraph("Soil Quality Report - Polluted Soils Only", styles['Title']))
    elements.append(Spacer(1, 12))
    legend_text = """
    <b>Legend:</b><br/>
    <font color='red'>Red</font> → Problematic value / Polluted status<br/>
    <font color='green'>Green</font> → Healthy value / Unpolluted status
    """
    elements.append(Paragraph(legend_text, styles['Normal']))
    elements.append(PageBreak())

    if df_polluted.empty:
        elements.append(Paragraph("✅ No polluted soils found.", styles['Normal']))
    else:
        feature_thresholds = {
            "pH_real": {"low": 6.5, "high": 8},
            "nitrate_real": {"low": 0, "high": 0.8},
            "moisture_real": {"low": 0.3, "high": 1.0},
            "organic_content_real": {"low": 0.2, "high": 1.0},
            "temperature_real": {"low": 0, "high": 35}
        }

        for _, row in df_polluted.iterrows():
            elements.append(Paragraph(f"Soil Report - {row['sample_id']}", styles['Heading2']))
            elements.append(Paragraph(f"<b>Location:</b> {row['location']} | <b>Date:</b> {row['date']}", styles['Normal']))
            elements.append(Spacer(1, 12))

            elements.append(Paragraph(f"<b>Status:</b> <font color='red'>{row['Predicted_Label'].upper()}</font> "
                                      f"(Probability: {row['Predicted_Probability']:.2f})", styles['Normal']))
            elements.append(Spacer(1, 12))

            feature_data = [["Feature", "Value (Real-world)"]]
            for f in ["pH_real","nitrate_real","moisture_real","organic_content_real","temperature_real"]:
                val = row[f]
                thresholds = feature_thresholds[f]
                color = "red" if (
                    (f in ["pH_real","moisture_real","organic_content_real"] and val < thresholds["low"])
                    or (f in ["nitrate_real","temperature_real"] and val > thresholds["high"])
                ) else "green"
                feature_data.append([f.replace("_real","").capitalize(), Paragraph(f"<font color='{color}'>{val:.2f}</font>", styles["Normal"])])

            table = Table(feature_data, hAlign='LEFT')
            table.setStyle(TableStyle([
                ('BACKGROUND', (0,0), (-1,0), colors.lightgrey),
                ('GRID', (0,0), (-1,-1), 0.5, colors.grey)
            ]))
            elements.append(table)
            elements.append(Spacer(1, 12))
            elements.append(Paragraph(f"<b>Actions:</b> {row['Suggested_Actions']}", styles['Normal']))
            elements.append(PageBreak())

    pdf_filename = "Polluted_Soil_Reports.pdf"
    doc = SimpleDocTemplate(pdf_filename, pagesize=A4)
    doc.build(elements, onFirstPage=add_page_number, onLaterPages=add_page_number)
    return pdf_filename
